# Radiology Knowledge Graph Extraction with CheXpert Plus (ReXKG)

Full **local** walkthrough of the ReXKG pipeline:

1. Configure local paths and load PyHealth from the repo
2. Load the CheXpert Plus dataset (`CheXpertPlusDataset`)
3. Apply the KG extraction task (`RadiologyKGExtractionTask`)
4. **GPT-4o entity extraction** on a small subset (mirrors `gpt4_entity_extraction.py`)
5. **GPT-4o relation extraction** (mirrors `gpt4_relation_extraction.py`)
6. Initialize `ReXKGModel` (BERT span-NER + pairwise RE)
7. Build and inspect the Knowledge Graph

**Papers:** ReXKG https://arxiv.org/abs/2408.14397 · CheXpert Plus https://arxiv.org/abs/2405.19111  
**Authors:** Aaron Miller · Kathryn Thompson · Pushpendra Tiwari

## Step 0: Install dependencies

Run once per environment. PyHealth is loaded from the repo — no pip install needed for it.

In [ ]:
%pip install --quiet transformers ipywidgets openai tqdm networkx matplotlib pandas torch


## Step 1: Configure local paths

Expected layout:
```
ReXKG-main/
├── PyHealth/          ← forked repo (contains the pyhealth/ package)
│   └── examples/cxr/  ← this notebook
└── src/
    └── data/chexpert_plus/   ← df_chexpert_plus_240401.csv
```
The cell walks up from `os.getcwd()` until it finds a `pyhealth/` sub-directory,
so it works regardless of how the Jupyter kernel was launched.

In [ ]:
import sys, os

# Walk up from cwd to find the PyHealth repo root (the dir that contains pyhealth/)
_search = os.path.abspath(os.getcwd())
PYHEALTH_DIR = None
for _ in range(6):
    if os.path.isdir(os.path.join(_search, 'pyhealth')):
        PYHEALTH_DIR = _search
        break
    _search = os.path.dirname(_search)
if PYHEALTH_DIR is None:
    raise RuntimeError(
        f"Could not find a 'pyhealth/' package starting from {os.getcwd()}.\n"
        "Launch the kernel from inside the PyHealth repo (e.g. cd PyHealth && jupyter notebook)."
    )

DATA_ROOT = os.path.abspath(os.path.join(PYHEALTH_DIR, '..', 'src', 'data', 'chexpert_plus'))
CACHE_DIR  = os.path.join(PYHEALTH_DIR, 'examples', 'cxr', 'rexkg_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

if PYHEALTH_DIR not in sys.path:
    sys.path.insert(0, PYHEALTH_DIR)

import importlib, pyhealth
importlib.reload(pyhealth)
print('pyhealth loaded from :', pyhealth.__file__)
print('DATA_ROOT            :', DATA_ROOT)
print('CACHE_DIR            :', CACHE_DIR)
assert os.path.isdir(DATA_ROOT), (
    f'DATA_ROOT not found: {DATA_ROOT}\n'
    'Place df_chexpert_plus_240401.csv inside that folder.'
)


## Step 2: Load the CheXpert Plus Dataset

`path_to_image` is used as the patient identifier; `section_findings` is the
radiology report text.  `dev=True` caps at 1 000 studies for exploration.

In [ ]:
from pyhealth.datasets import CheXpertPlusDataset

dataset = CheXpertPlusDataset(
    root=DATA_ROOT,
    dev=True,
)
dataset.stats()


## Step 3: Apply the Radiology KG Extraction Task

`RadiologyKGExtractionTask` converts each report into a sample:  
`{text, entities: [], relations: []}` — entity/relation lists are populated at inference.

In [ ]:
from pyhealth.tasks import RadiologyKGExtractionTask

task = RadiologyKGExtractionTask(
    findings_only=True,
    min_text_length=10,
)
samples = dataset.set_task(task)
print(f'Total samples: {len(samples)}')


## Step 4: GPT-4o Entity Extraction (small subset)

Mirrors `gpt4_entity_extraction.py`.  
Entity types: `anatomy`, `disorder_present`, `disorder_notpresent`, `procedures`,
`devices_present`, `devices_notpresent`, `size`, `concept`.

> **Cost:** ~\$0.01–0.03 per report with `gpt-4o`. Keep `N_SAMPLES` small (5–20) for exploration.

> **Security:** Set `OPENAI_API_KEY` as an environment variable — never paste the key directly in this cell.

In [ ]:
import os

# Set via shell: export OPENAI_API_KEY='sk-...'
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY',  '')
OPENAI_API_BASE = os.getenv('OPENAI_API_BASE', 'https://api.openai.com/v1')

N_SAMPLES = 5  # reports to send to GPT

GPT_ENTITY_CACHE   = os.path.join(CACHE_DIR, 'gpt4_entities_subset.json')
GPT_RELATION_CACHE = os.path.join(CACHE_DIR, 'gpt4_relations_subset.json')
print('Entity cache  :', GPT_ENTITY_CACHE)
print('Relation cache:', GPT_RELATION_CACHE)


In [ ]:
import json, time
import openai
from tqdm import tqdm

client = openai.OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_API_BASE)

# ── Few-shot prompt (from gpt4_entity_extraction.py) ────────────────────── #
_NER_FEWSHOT = [
    {
        'context': (
            '<Input> Unchanged position of the left upper extremity PICC line. '
            'Again seen are surgical clips projecting over the right hemithorax. '
            'Increased stranding opacities are noted in the left retrocardiac region.<\\Input>'
        ),
        'response': (
            "{'Unchanged position of the left upper extremity PICC line.':"
            "{'Unchanged':'concept','position':'concept','left':'concept','upper':'concept',"
            "'extremity':'anatomy','PICC line':'device_present'},"
            "'Again seen are surgical clips projecting over the right hemithorax.':"
            "{'surgical clips':'device_present','right':'concept','hemithorax':'anatomy'},"
            "'Increased stranding opacities are noted in the left retrocardiac region.':"
            "{'Increased':'concept','stranding':'concept','opacities':'disorder_present',"
            "'left':'concept','retrocardiac':'anatomy','region':'anatomy'}}"
        ),
    }
]

_NER_SYSTEM = (
    'You are a radiologist performing clinical term extraction from the FINDINGS and '
    'IMPRESSION sections in the radiology report. '
    "A clinical term can be in ['anatomy','disorder_present','disorder_notpresent',"
    "'procedures','devices','concept','devices_present','devices_notpresent','size']. "
    "'anatomy'=body part; 'disorder_present'=present finding; 'disorder_notpresent'=absent finding; "
    "'procedures'=diagnostic/therapeutic procedure; 'devices'=medical apparatus; "
    "'size'=measurement e.g. 3mm; 'concept'=descriptor/modifier. "
    'Extract one word at a time. Words like "no" are NOT entities. '
    'Input: <Input><sentence><sentence><\\Input>. '
    "Reply JSON: {'<sentence>':{'entity':'entity_type',...},...}"
)


def _ner_messages(text):
    msgs = [{'role': 'system', 'content': _NER_SYSTEM}]
    for s in _NER_FEWSHOT:
        msgs.append({'role': 'user',      'content': s['context']})
        msgs.append({'role': 'assistant', 'content': s['response']})
    msgs.append({'role': 'user', 'content': f'<Input> {text} <\\Input>'})
    return msgs


def _estimate_cost(prompt_tokens, completion_tokens):
    return prompt_tokens * 0.005 / 1000 + completion_tokens * 0.015 / 1000


def gpt_extract_entities(text):
    resp = client.chat.completions.create(
        model='gpt-4o-2024-05-13',
        messages=_ner_messages(text),
        response_format={'type': 'json_object'},
    )
    raw  = resp.choices[0].message.content
    cost = _estimate_cost(resp.usage.prompt_tokens, resp.usage.completion_tokens)
    try:
        return json.loads(raw), cost
    except Exception:
        return raw, cost


print('GPT entity-extraction helpers ready.')


In [ ]:
import pandas as pd

csv_path = os.path.join(DATA_ROOT, 'df_chexpert_plus_240401.csv')
df_gpt = pd.read_csv(csv_path).dropna(subset=['section_findings']).head(N_SAMPLES)

try:
    with open(GPT_ENTITY_CACHE) as fh:
        entity_cache = json.load(fh)
    print(f'Loaded {len(entity_cache)} cached entity results.')
except FileNotFoundError:
    entity_cache = {}

total_cost = 0.0
for _, row in tqdm(df_gpt.iterrows(), total=len(df_gpt), desc='GPT entity extraction'):
    pid, text = row['path_to_image'], row['section_findings']
    if pid in entity_cache:
        continue
    try:
        res, cost = gpt_extract_entities(text)
        total_cost += cost
        entity_cache[pid] = {'section_findings': text, 'res': res, 'cost': cost}
    except Exception as exc:
        print(f'  Error on {pid}: {exc}')
        time.sleep(1)
    with open(GPT_ENTITY_CACHE, 'w') as fh:
        json.dump(entity_cache, fh, indent=2)

print(f'\nDone. Cumulative cost: ${total_cost:.4f}')
print(f'Saved to {GPT_ENTITY_CACHE}')


In [ ]:
# Inspect entity results for the first study
first_key = next(iter(entity_cache))
print('Patient/Study:', first_key)
print('Findings snippet:', entity_cache[first_key]['section_findings'][:200])
print('\nExtracted entities (per sentence):')
for sent, ents in entity_cache[first_key]['res'].items():
    print(f'  [{sent[:60]}...]')
    for term, etype in ents.items():
        print(f'    {term!r:30s} -> {etype}')


## Step 5: GPT-4o Relation Extraction

Mirrors `gpt4_relation_extraction.py`.  
Relation types: `modify`, `located_at`, `suggestive_of`.

In [ ]:
# ── Few-shot prompt (from gpt4_relation_extraction.py) ──────────────────── #
_RE_FEWSHOT = [
    {
        'context': (
            "{'Bones are stable with mild degenerative changes of the spine.':"
            "{'Bones':'anatomy','stable':'concept','mild':'concept',"
            "'degenerative changes':'disorder_present','spine':'anatomy'}}"
        ),
        'response': (
            "{'Bones are stable with mild degenerative changes of the spine.':"
            "[{'stable':'Bones','relation':'modify'},"
            "{'mild':'degenerative changes','relation':'modify'},"
            "{'degenerative changes':'spine','relation':'located_at'}]}"
        ),
    },
    {
        'context': (
            "{'A dense retrocardiac opacity remains present with slight blunting of "
            "the left costophrenic angle, suggestive of a small effusion.':"
            "{'dense':'concept','retrocardiac':'anatomy','opacity':'disorder_present',"
            "'slight':'concept','blunting':'disorder_present','left':'concept',"
            "'costophrenic':'anatomy','angle':'anatomy','small':'concept','effusion':'disorder_present'}}"
        ),
        'response': (
            "{'A dense retrocardiac opacity remains present with slight blunting of "
            "the left costophrenic angle, suggestive of a small effusion.':"
            "[{'dense':'opacity','relation':'modify'},{'opacity':'retrocardiac','relation':'located_at'},"
            "{'slight':'blunting','relation':'modify'},{'blunting':'angle','relation':'modify'},"
            "{'left':'costophrenic','relation':'modify'},{'small':'effusion','relation':'modify'},"
            "{'effusion':'costophrenic','relation':'located_at'},"
            "{'opacity':'effusion','relation':'suggestive_of'},"
            "{'blunting':'effusion','relation':'suggestive_of'}]}"
        ),
    },
]

_RE_SYSTEM = (
    'You are a radiologist performing relation extraction from radiology reports. '
    "Relations: 'modify' (source modifies target), 'located_at' (source located at anatomy), "
    "'suggestive_of' (source finding suggests target disease). "
    "concept->anatomy links always use 'modify'. "
    "Input JSON: {'sentence':{'entity':'type',...},...}. "
    "Reply JSON: {'sentence':[{'src':'tgt','relation':'rel'},...]}."
)


def _re_messages(entity_json):
    msgs = [{'role': 'system', 'content': _RE_SYSTEM}]
    for s in _RE_FEWSHOT:
        msgs.append({'role': 'user',      'content': s['context']})
        msgs.append({'role': 'assistant', 'content': s['response']})
    msgs.append({'role': 'user', 'content': entity_json})
    return msgs


def gpt_extract_relations(entity_dict):
    resp = client.chat.completions.create(
        model='gpt-4o-2024-05-13',
        messages=_re_messages(json.dumps(entity_dict)),
        response_format={'type': 'json_object'},
    )
    raw  = resp.choices[0].message.content
    cost = _estimate_cost(resp.usage.prompt_tokens, resp.usage.completion_tokens)
    try:
        return json.loads(raw), cost
    except Exception:
        return raw, cost


print('GPT relation-extraction helpers ready.')


In [ ]:
try:
    with open(GPT_RELATION_CACHE) as fh:
        relation_cache = json.load(fh)
    print(f'Loaded {len(relation_cache)} cached relation results.')
except FileNotFoundError:
    relation_cache = {}

total_re_cost = 0.0
for pid, entry in tqdm(entity_cache.items(), desc='GPT relation extraction'):
    if pid in relation_cache:
        continue
    entity_dict = entry.get('res', {})
    if not entity_dict:
        continue
    try:
        res, cost = gpt_extract_relations(entity_dict)
        total_re_cost += cost
        relation_cache[pid] = {**entry, 'res_relation': res, 'cost_relation': cost}
    except Exception as exc:
        print(f'  Error on {pid}: {exc}')
        time.sleep(1)
    with open(GPT_RELATION_CACHE, 'w') as fh:
        json.dump(relation_cache, fh, indent=2)

print(f'\nDone. Cumulative relation cost: ${total_re_cost:.4f}')
print(f'Saved to {GPT_RELATION_CACHE}')


In [ ]:
# Inspect relation results for the first study
first_key = next(iter(relation_cache))
print('Patient/Study:', first_key)
print('\nExtracted relations (per sentence):')
for sent, rels in relation_cache[first_key].get('res_relation', {}).items():
    print(f'  [{sent[:60]}...]')
    if isinstance(rels, list):
        for r in rels:
            rel_type = r.get('relation', '?')
            entities = [(k, v) for k, v in r.items() if k != 'relation']
            if entities:
                src, tgt = entities[0]
                print(f"    '{src}' --[{rel_type}]--> '{tgt}'")


## Step 6: Initialize the ReXKG Model (BERT-based)

`ReXKGModel` attaches a **span NER head** and a **pairwise relation head** on top of a BERT encoder.  
Use for large-scale inference after fine-tuning, or combine with the GPT outputs above.

In [ ]:
from pyhealth.models import ReXKGModel

model = ReXKGModel(
    dataset=samples,
    bert_model_name='bert-base-uncased',
    max_span_length=4,  # smaller -> fewer candidate spans -> faster pairwise RE
)
print(model)


### (Optional) Load a pretrained ReXKG checkpoint

If you have a checkpoint fine-tuned on ReXKG / MIMIC NER, load it here.  
The **GPT-extracted** entities/relations from Steps 4–5 can also be used directly to build a KG without fine-tuning.

In [ ]:
# Load the pretrained BERT encoder weights from src/ner/result/run_entity/
# Only the BERT backbone is loaded (head shapes differ); task heads remain randomly initialized.
from safetensors.torch import load_file as load_safetensors
import os

ENTITY_CKPT = os.path.abspath(
    os.path.join(PYHEALTH_DIR, '..', 'src', 'ner', 'result', 'run_entity', 'model.safetensors')
)

if os.path.exists(ENTITY_CKPT):
    ckpt = load_safetensors(ENTITY_CKPT)

    # Remap BERT backbone keys: 'bert.*' -> 'encoder.*'; skip all task-head keys
    remapped = {
        'encoder.' + k[5:]: v
        for k, v in ckpt.items()
        if k.startswith('bert.')
    }

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    print(f'Loaded encoder weights from: {ENTITY_CKPT}')
    print(f'  BERT layers loaded : {len(remapped)}')
    print(f'  Missing (task heads, remain random): {len(missing)}')
else:
    print(f'Checkpoint not found at {ENTITY_CKPT} — using random weights.')


## Step 6b: BERT-based Entity & Relation Extraction

Runs on the same small subset sent to GPT — useful for direct comparison.

In [ ]:
# Use only the first 2 reports for BERT demo (heads are randomly initialized)
BERT_DEMO_N = 2

reports     = [v['section_findings'] for v in entity_cache.values()][:BERT_DEMO_N]
patient_ids = list(entity_cache.keys())[:BERT_DEMO_N]

bert_entities = model.predict_entities(reports, batch_size=2)
for pid, ents in zip(patient_ids, bert_entities):
    print(f'\n{pid}: {len(ents)} entities')
    for e in ents:
        print(f"  [{e['type']}] '{e['text']}'")


In [ ]:
bert_relations = model.predict_relations(reports, bert_entities, batch_size=2)
for pid, rels in zip(patient_ids, bert_relations):
    print(f'\n{pid}: {len(rels)} relations')
    for r in rels:
        print(f"  '{r['subject']['text']}' --[{r['relation']}]--> '{r['object']['text']}'")


## Step 7: Build and Inspect the Knowledge Graph

`build_kg()` assembles a deduplicated KG:
- `nodes`: entity nodes (id, text, type)
- `edges`: relation triples (subject_id, relation, object_id)
- `subgraphs`: per-study subgraph

Demonstrated with BERT outputs below. Swap in `entity_cache` / `relation_cache` from Steps 4–5 to use GPT results.

In [ ]:
kg = model.build_kg(reports=reports, patient_ids=patient_ids)

print(f"KG nodes : {len(kg['nodes'])}")
print(f"KG edges : {len(kg['edges'])}")

print('\nNodes:')
for n in kg['nodes']:
    print(f"  [{n['id']}] {n['text']}  ({n['type']})")

print('\nEdges:')
node_map = {n['id']: n['text'] for n in kg['nodes']}
for e in kg['edges']:
    print(f"  '{node_map[e['subject_id']]}' --[{e['relation']}]--> '{node_map[e['object_id']]}'")


In [ ]:
kg_out = os.path.join(CACHE_DIR, 'chexpert_plus_kg_demo.json')
model.save_kg(kg, kg_out)
print(f'KG saved to {kg_out}')


## Step 8: (Optional) Visualize the Subgraph

In [ ]:
try:
    import networkx as nx
    import matplotlib.pyplot as plt

    TOP_N_NODES = 100
# show only the top-N most-connected nodes

    G_full = nx.DiGraph()
    node_map = {n['id']: n['text'] for n in kg['nodes']}
    for e in kg['edges']:
        G_full.add_edge(node_map[e['subject_id']], node_map[e['object_id']], label=e['relation'])

    # Keep only the TOP_N_NODES nodes with the highest degree
    top_nodes = sorted(G_full.degree, key=lambda x: x[1], reverse=True)[:TOP_N_NODES]
    top_node_names = {n for n, _ in top_nodes}
    G = G_full.subgraph(top_node_names).copy()

    pos = nx.spring_layout(G, seed=42, k=1.5)
    edge_labels = nx.get_edge_attributes(G, 'label')

    plt.figure(figsize=(14, 9))
    nx.draw(G, pos, with_labels=True, node_size=1800, node_color='lightblue',
            font_size=8, arrows=True, arrowsize=15)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
    plt.title(f'ReXKG Knowledge Graph - Top {TOP_N_NODES} nodes (CheXpert Plus Demo)')
    plt.tight_layout()
    plt.show()
    print(f'Showing {G.number_of_nodes()} nodes, {G.number_of_edges()} edges '
          f'(full graph: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges)')
except ImportError:
    print('pip install networkx matplotlib')


## Summary

| Step | Component | What it does |
|------|-----------|-------------|
| 1 | Path setup | Auto-locates `pyhealth/` package; sets `DATA_ROOT` and `CACHE_DIR` |
| 2 | `CheXpertPlusDataset` | Loads CheXpert Plus CSV; one study per row |
| 3 | `RadiologyKGExtractionTask` | Converts findings text into structured samples |
| 4 | GPT-4o entity extraction | Few-shot NER on small subset; cached to JSON |
| 5 | GPT-4o relation extraction | Few-shot RE on entity dict; cached to JSON |
| 6 | `ReXKGModel` | BERT encoder + span NER head + pairwise RE head |
| 6b | `predict_entities/relations` | BERT-based inference at scale |
| 7 | `build_kg` / `save_kg` | Assembles and serialises deduplicated KG |
| 8 | NetworkX | Optional subgraph visualisation |
